# Task 3 — Usage U3 two-stage screen

Run All trains the SmallCNN from scratch on canonical folds 0 and 4. First learn image features for 30 epochs with ordinary cross-entropy. Then freeze those features and train a new balanced classifier for 10 epochs.

Use a **fresh Colab L4 GPU session** with the existing Task 3 runtime. No local PyTorch installation is needed. The only scored model is Stage B epoch 10.


## 1. Colab GPU and repository


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import time
import zipfile

REPO_URL = "https://github.com/TrnLin/MLA2.git"
BRANCH = "fashion-analysis-and-cleanup"
CHECKOUT_DIR = Path("/content/MLA2")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT_DIR = DRIVE_MOUNT / "MyDrive/MLA2"
DATA_ZIP = DRIVE_PROJECT_DIR / "data/task3-data.zip"
LOCAL_DATA_ZIP = Path("/content/task3-data.zip")
DRIVE_TASK_DIR = DRIVE_PROJECT_DIR / "task3"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "results/runs.csv"

def run_checked(command, *, cwd=None):
    command = [str(part) for part in command]
    print("$", " ".join(command), flush=True)
    return subprocess.run(command, cwd=cwd, check=True)

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Connect this notebook to a Google Colab GPU runtime first.") from exc

drive.mount(str(DRIVE_MOUNT), force_remount=False)
if (CHECKOUT_DIR / ".git").is_dir():
    remote_url = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=CHECKOUT_DIR, text=True
    ).strip()
    if remote_url != REPO_URL:
        raise RuntimeError(f"{CHECKOUT_DIR} belongs to a different repository: {remote_url}")
    run_checked(["git", "fetch", "origin", BRANCH], cwd=CHECKOUT_DIR)
    run_checked(["git", "switch", BRANCH], cwd=CHECKOUT_DIR)
    run_checked(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=CHECKOUT_DIR)
elif CHECKOUT_DIR.exists():
    raise RuntimeError(f"{CHECKOUT_DIR} exists but is not a Git repository.")
else:
    run_checked(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, CHECKOUT_DIR])

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT_DIR, text=True).strip()
REPO_DIR = CHECKOUT_DIR / "core" if (CHECKOUT_DIR / "core/src/fashion").is_dir() else CHECKOUT_DIR
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
print("Repository ready:", REPO_DIR)
print("Commit:", commit)


Mounted at /content/drive
$ git clone --branch task-3-gender-usage-classification --single-branch https://github.com/TrnLin/MLA2.git /content/MLA2
Repository ready: /content/MLA2
Commit: 67e71e5cc762fcc2573f3a215c1f43ffd578b041


## 2. Teacher data and canonical split


In [2]:
def copy_teacher_zip_to_local_disk():
    if LOCAL_DATA_ZIP.is_file():
        try:
            with zipfile.ZipFile(LOCAL_DATA_ZIP) as existing:
                existing.infolist()
            return
        except zipfile.BadZipFile:
            LOCAL_DATA_ZIP.unlink()
    partial = LOCAL_DATA_ZIP.with_suffix(".zip.partial")
    for attempt in range(1, 4):
        partial.unlink(missing_ok=True)
        try:
            expected_bytes = DATA_ZIP.stat().st_size
            with DATA_ZIP.open("rb") as source, partial.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)
            if partial.stat().st_size != expected_bytes:
                raise OSError("The local ZIP copy is incomplete.")
            partial.replace(LOCAL_DATA_ZIP)
            return
        except OSError as error:
            partial.unlink(missing_ok=True)
            if attempt == 3:
                raise RuntimeError("Drive disconnected three times. Remount and retry.") from error
            drive.mount(str(DRIVE_MOUNT), force_remount=True)
            time.sleep(2)

copy_teacher_zip_to_local_disk()
teacher_dir = REPO_DIR / "data/raw/teacher"
required_files = (
    teacher_dir / "train/styles_train.csv",
    teacher_dir / "test/styles_prediction.csv",
)
image_dirs = (teacher_dir / "train/images_train", teacher_dir / "test/images_test")
image_suffixes = {".jpg", ".jpeg"}
with zipfile.ZipFile(LOCAL_DATA_ZIP) as archive:
    names = archive.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("The teacher archive contains an unsafe path.")
    expected_images = sum(
        name.startswith("data/raw/teacher/") and Path(name).suffix.lower() in image_suffixes
        for name in names
    )
    current_images = sum(
        path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
    )
    if current_images != expected_images or not all(path.is_file() for path in required_files):
        archive.extractall(REPO_DIR)

actual_images = sum(
    path.suffix.lower() in image_suffixes for folder in image_dirs for path in folder.glob("*")
)
if actual_images != expected_images or not all(path.is_file() for path in required_files):
    raise RuntimeError(f"Teacher data is incomplete: {actual_images:,}/{expected_images:,} images")

os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
(DRIVE_TASK_DIR / "results").mkdir(parents=True, exist_ok=True)
print(f"Teacher data ready: {actual_images:,} images")


Teacher data ready: 44,441 images


## 3. Fixed recipe and source check

Keep 80×60 RGB, widths `[32, 64, 128, 256]`, average pooling, no dropout and no augmentation. Keep all nine classes, including literal `NA` and `Home`.

Stage A: 30 epochs, ordinary cross-entropy. Stage B: freeze every feature weight and BatchNorm buffer, cache only outer-training features, reset the nine-class head with seed 2753, then train it for 10 epochs with E2's effective-number weights (beta 0.999, cap 5). Each training row appears once per epoch.

Each stage starts a fresh AdamW optimizer at 0.001, weight decay 0.0001, with its own cosine schedule ending at 0.00001. No early stopping or checkpoint selection. Stage A predictions are saved to explain the result.

The check below verifies the two fixed E2 source bundles before any training.


In [3]:
from fashion.train.task3_usage_two_stage import (
    check_usage_two_stage_sources, run_usage_two_stage_screen, recipe,
)

E2_DIR = DRIVE_TASK_DIR / "experiments/t3_usage_e2_class_balanced_ce/usage"
sources, splits = check_usage_two_stage_sources(
    e2_directory=E2_DIR, registry_path=DRIVE_REGISTRY, root=REPO_DIR,
)
print("E2 sources verified:", {fold: source["run_id"] for fold, source in sources.items()})
print(recipe())


E2 sources verified: {0: 't3_usage_e2_class_balanced_ce_usage_smallcnn_f0_s2753_5461e048c3b3_20260830T115815Z356f6d', 4: 't3_usage_e2_class_balanced_ce_usage_smallcnn_f4_s2753_5461e048c3b3_20260830T123218Z94db47'}
{'experiment_id': 't3_usage_u3_two_stage_cnn', 'baseline': {'target': 'usage', 'image_height': 80, 'image_width': 60, 'channels': [32, 64, 128, 256], 'batch_size': 128, 'epochs': 30, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'minimum_learning_rate': 1e-05, 'seed': 2753, 'num_workers': 2, 'mixed_precision': False, 'early_stopping': False, 'augmentation': 'none', 'loss_name': 'cross_entropy', 'optimizer_name': 'AdamW', 'scheduler_name': 'CosineAnnealingLR', 'checkpoint_rule': 'final_epoch', 'model_family': 'task3_small_cnn', 'scratch': True, 'submission_eligible': True, 'num_classes': 9}, 'stage_a_epochs': 30, 'stage_a_loss': 'cross_entropy', 'stage_b_epochs': 10, 'stage_b_loss': 'effective_number_cross_entropy', 'stage_b_beta': 0.999, 'stage_b_cap': 5.0, 'stage_b_reset_s

## 4. Run the two-fold screen

Pass requires either pooled macro-F1 ≥ 0.417319 with a positive lower 95% paired-family bootstrap bound, or macro-F1 ≥ 0.402319 with at least 10% better NLL or Brier score than E2. NLL and Brier measure probability errors.

Both routes also require ECE ≤ 0.05, no non-Home class F1 drop over 0.03, rare-class predictions ≤ 5 times support in each fold and pooled, and no mean corruption-induced drop over 0.02 relative to E2 for any of five checks. The clean training-gap route stays unavailable until matching E2 training scores exist.

Each fold gets **90 minutes including setup and diagnostics**, with **16 GiB host RAM and 7 GiB GPU allocator memory**. A watchdog checks process-tree RAM every 0.2 seconds; brief overshoots remain possible. Failure stops the screen before the next fold. Every fit is registered before training starts.

Complete saved runs are verified and reused. Failed runs start fresh; neither stage resumes from a partial checkpoint.


In [4]:
result = run_usage_two_stage_screen(
    e2_directory=E2_DIR, source_registry_path=DRIVE_REGISTRY,
    output_root=DRIVE_TASK_DIR, registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,), root=REPO_DIR,
)
print("Screen:", result["status"])
for item in result["checks"]:
    print(item)
print("Saved:", DRIVE_TASK_DIR / "experiments/t3_usage_u3_two_stage_cnn/usage/screen_decision.json")


Usage fold 0: training_only_normalization 
Usage fold 0: stage_a 1
loss=0.615963; validation CE=0.656647
Usage fold 0: stage_a 2
loss=0.425828; validation CE=0.453803
Usage fold 0: stage_a 3
loss=0.384201; validation CE=0.464684
Usage fold 0: stage_a 4
loss=0.355970; validation CE=0.437220
Usage fold 0: stage_a 5
loss=0.330416; validation CE=0.591485
Usage fold 0: stage_a 6
loss=0.315146; validation CE=0.439216
Usage fold 0: stage_a 7
loss=0.290593; validation CE=0.375235
Usage fold 0: stage_a 8
loss=0.275295; validation CE=1.848168
Usage fold 0: stage_a 9
loss=0.253184; validation CE=1.560946
Usage fold 0: stage_a 10
loss=0.236530; validation CE=0.770135
Usage fold 0: stage_a 11
loss=0.216937; validation CE=0.998842
Usage fold 0: stage_a 12
loss=0.195943; validation CE=0.892969
Usage fold 0: stage_a 13
loss=0.175926; validation CE=0.422054
Usage fold 0: stage_a 14
loss=0.154284; validation CE=0.647509
Usage fold 0: stage_a 15
loss=0.135027; validation CE=1.426736
Usage fold 0: stage_a

## 5. Stop and review

Review the saved Stage A and Stage B predictions, clean training scores, class scores, corruption checks and screen decision. This is development evidence; repeated use of these folds can bias model choice.

No extra folds, seeds, refit or held-out test runs start automatically. Tell me when it finishes so we can check the result.
